In [1]:
import pandas as pd
from collections import Counter
import random
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import os

In [2]:
random.seed(1618)

In [3]:
dataset_list = ['dementiaAudio', 'huffPostNews', 'medicalAbstracts']

In [4]:
zero_dfs = []
rule_dfs = []
text_dfs = []
for folder in os.listdir(f'../mda_twoBest_thisExampleIsBLANK_theWritingStyleOfThisTextIsBLANK/outputsAll/'):
    if os.path.isdir(f'../mda_twoBest_thisExampleIsBLANK_theWritingStyleOfThisTextIsBLANK/outputsAll/{folder}'):
        zero_dfs.append(pd.read_csv(f'../mda_twoBest_thisExampleIsBLANK_theWritingStyleOfThisTextIsBLANK/outputsAll/{folder}/mean_model_zero_shot_classification.csv').drop(columns=['factor_1_label', 'factor_2_label', 'factor_3_label', 'factor_4_label', 'factor_5_label', 'factor_6_label']))
        rule_dfs.append(pd.read_csv(f'../mda_twoBest_thisExampleIsBLANK_theWritingStyleOfThisTextIsBLANK/outputsAll/{folder}/mda_dim_scores.csv').drop(columns=['doc_cat', 'factor_1_label', 'factor_2_label', 'factor_3_label', 'factor_4_label', 'factor_5_label', 'factor_6_label', 'factor_7_label', 'factor_7']))
        text_dfs.append(pd.read_csv(f'../mda_twoBest_thisExampleIsBLANK_theWritingStyleOfThisTextIsBLANK/outputsAll/{folder}/texts_and_ids.csv'))

In [5]:
zero_df = pd.concat(zero_dfs)
rule_df = pd.concat(rule_dfs)
text_df = pd.concat(text_dfs)

for column in zero_df.columns:
    if 'factor' in column:
        zero_df[f'{column}_zero'] = zero_df[column]
        rule_df[f'{column}_rule'] = rule_df[column]
        zero_df = zero_df.drop(columns=[column])
        rule_df = rule_df.drop(columns=[column])

In [6]:
zero_df

,doc_id,factor_1_zero,factor_2_zero,factor_3_zero,factor_4_zero,factor_5_zero,factor_6_zero
0,atis_2010,-1.200341,-0.989405,0.923170,-0.204761,-0.235638,0.688347
1,atis_2201,0.119553,-0.261426,0.557411,0.442611,-0.529781,1.069579
2,atis_2319,-0.072308,-0.692275,0.175982,0.017557,-0.232144,1.121618
3,atis_2635,0.449365,-0.284083,0.470683,0.144500,-0.382803,1.402498
4,atis_266,0.490909,0.047149,-1.116249,1.002571,-0.424236,1.531107
...,...,...,...,...,...,...,...
95,yahoo_69044,0.019144,0.670542,0.233626,1.276614,0.714673,0.361635
96,yahoo_75331,-0.500308,-0.443670,-0.323815,-1.320123,2.758217,-1.476097
97,yahoo_78207,-0.156032,-2.150551,1.030487,1.597719,-0.158387,1.556374
98,yahoo_82858,1.053531,-1.188108,-0.464690,-0.638542,0.637179,-1.388866


In [7]:
merged = pd.merge(zero_df, rule_df, on='doc_id')
merged_df = pd.merge(text_df, merged, on='doc_id')

In [8]:
merged_df

,text,doc_id,category,factor_1_zero,factor_2_zero,factor_3_zero,factor_4_zero,factor_5_zero,factor_6_zero,factor_1_rule,factor_2_rule,factor_3_rule,factor_4_rule,factor_5_rule,factor_6_rule
0,can you give me the evening flight on wednesda...,atis_310,atis,0.609205,0.424999,-0.229155,-0.208389,0.080600,0.571829,0.025366,-0.237068,-2.513959,0.965014,-0.349251,-0.378634
1,list all flights going from boston to atlanta ...,atis_2010,atis,-1.200341,-0.989405,0.923170,-0.204761,-0.235638,0.688347,-0.871803,-0.753147,0.133903,-0.867581,2.259774,-0.378634
2,what flights from kansas city to denver after ...,atis_2319,atis,-0.072308,-0.692275,0.175982,0.017557,-0.232144,1.121618,-1.210686,-0.237068,0.133903,-0.867581,-0.349251,-0.378634
3,please show me flights from san francisco to d...,atis_266,atis,0.490909,0.047149,-1.116249,1.002571,-0.424236,1.531107,-0.529022,-0.237068,0.133903,-0.867581,-0.349251,-0.378634
4,i'd like the earliest flight from dallas to bo...,atis_3595,atis,0.439930,0.015732,0.575234,0.325555,-0.628409,1.406406,-0.533157,-0.769964,0.133903,1.404560,-0.349251,-0.378634
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23627,"When you are truly happy, when you can not wai...",yahoo_9294,yahoo,1.314066,2.274550,-0.991503,1.292247,-1.571217,2.097301,2.858028,-0.626792,0.241301,2.286884,-0.352837,1.683455
23628,That dude that was talkin bout the alternator....,yahoo_41193,yahoo,-0.424514,0.412456,-1.456625,0.431616,0.371215,-0.804282,1.644665,0.170223,-1.706571,0.518006,1.291373,0.692678
23629,just click forward...and then delete what you ...,yahoo_82858,yahoo,1.053531,-1.188108,-0.464690,-0.638542,0.637179,-1.388866,1.840831,-0.460148,-1.753905,-0.723345,-0.352837,-0.351750
23630,Aerial roots are roots that are formed in and ...,yahoo_78207,yahoo,-0.156032,-2.150551,1.030487,1.597719,-0.158387,1.556374,-0.096578,-0.813646,0.091475,-0.723345,-0.352837,-0.351750


In [9]:
rows_to_save = []
score_type = []
score_type_no_method = []
high_or_low = []
for column in merged_df.columns:
    if 'factor' in column:
        # print(merged_df[merged_df[column] == merged_df[column].max()])
        rows_to_save.append(merged_df.loc[merged_df[column].idxmax()])
        score_type.append(column)
        score_type_no_method.append(column.replace('_zero', '').replace('_rule', ''))
        high_or_low.append('highest')
        rows_to_save.append(merged_df.loc[merged_df[column].idxmin()])
        score_type.append(column)
        score_type_no_method.append(column.replace('_zero', '').replace('_rule', ''))
        high_or_low.append('lowest')

df_to_save = pd.DataFrame(rows_to_save)
df_to_save['score_type'] = score_type
df_to_save['score_type_comparison'] = score_type_no_method
df_to_save['high_or_low'] = high_or_low

In [10]:
df_to_save.to_csv('./qualitativeAnalysis.csv')